# Colab Training Notebook - YOLO Traffic Detection

Notebook này dùng để chạy trên Google Colab:

1. Clone code Python từ GitHub.
2. Cấu hình Kaggle API.
3. Tải dataset từ Kaggle.
4. Train teacher YOLO và student YOLO baseline.
5. Xuất các file `.pt` cần dùng cho serving và báo cáo.

Lưu ý: Knowledge Distillation trong repo phải cần bản Ultralytics đã được chỉnh `ultralytics/ultralytics/engine/trainer.py` để nhận `teacher` và `distillation_config_loss`. Notebook này bật KD mặc định; nếu KD trainer chưa tương thích, cell KD sẽ báo lỗi để xử lý trực tiếp.

In [ ]:
# =====================
# 1. CONFIG
# =====================

GITHUB_REPO_URL = "https://github.com/hoaianthai345/HPC_Nhom1_MLOps_ObjectDectection.git"  # URL repo của nhóm, dùng để clone code vào Colab.
PROJECT_DIR = "/content/HPC_Nhom1_MLOps_ObjectDectection"  # Đường dẫn trong Colab sau khi clone repo.

# Fork Ultralytics KD. Notebook sẽ copy PROJECT_DIR/notebooks/trainer.py vào ultralytics/engine/trainer.py.
ULTRALYTICS_KD_REPO_URL = "https://github.com/ultralytics/ultralytics.git"
ULTRALYTICS_KD_DIR = "/content/ultralytics-kd"

KAGGLE_DATASET = "yusufberksardoan/traffic-detection-project"
RAW_DATA_DIR = "/content/data/raw"
SUBSET_DATA_DIR = "/content/data/demo_subset"
USE_SUBSET = True
SUBSET_SIZE = 500

# Train nhanh để kiểm tra pipeline: 5 epochs.
# Train báo cáo nên tăng lên 30-100 tùy GPU/time.
# Lệnh teacher đã kiểm chứng trên Colab T4: yolo26x.pt, batch=4.
# Student dùng yolo26n.pt để tương thích KD trainer dual-head one2many/one2one.
TEACHER_MODEL = "yolo26x.pt"
STUDENT_MODEL = "yolo26n.pt"
TEACHER_EPOCHS = 5
STUDENT_EPOCHS = 5
IMG_SIZE = 640
TEACHER_BATCH_SIZE = 4
STUDENT_BATCH_SIZE = 16

# Chạy Knowledge Distillation luôn. Nếu cell KD lỗi, cần sửa môi trường/KD trainer thay vì bỏ qua.
RUN_KD_OPTIONAL = True

ARTIFACT_DIR = "/content/model_artifacts"

print("Config loaded")

In [ ]:
# =====================
# 2. RUNTIME CHECK
# =====================

import os
import shutil
import subprocess
from pathlib import Path

def run(cmd, cwd=None, check=True):
    print(f"\$ {cmd}")
    return subprocess.run(cmd, shell=True, cwd=cwd, check=check)

try:
    import torch
    DEVICE = "0" if torch.cuda.is_available() else "cpu"
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception:
    DEVICE = "cpu"

print("Training device:", DEVICE)

## 3. Clone code từ GitHub

In [ ]:
if Path(PROJECT_DIR).exists():
    print(f"Repo already exists: {PROJECT_DIR}")
    run("git pull", cwd=PROJECT_DIR, check=False)
else:
    run(f"git clone {GITHUB_REPO_URL} {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("Current dir:", os.getcwd())
run("find . -maxdepth 2 -type f | sort | head -80", check=False)

## 4. Cài dependencies

In [ ]:
run("python -m pip install --upgrade pip")
run("pip install kaggle pyyaml mlflow boto3 botocore pillow opencv-python tqdm")

# Cài dependency của repo nếu file tồn tại. Không fail notebook nếu có package nặng không cần cho training cơ bản.
for req in ["data_pipeline/requirements.txt", "serving_pipeline/requirements.txt"]:
    if Path(req).exists():
        run(f"pip install -r {req}", check=False)

# Clone Ultralytics KD fork và thay trainer.py bằng bản custom của project.
if Path(ULTRALYTICS_KD_DIR).exists():
    run("git pull", cwd=ULTRALYTICS_KD_DIR, check=False)
else:
    run(f"git clone {ULTRALYTICS_KD_REPO_URL} {ULTRALYTICS_KD_DIR}")

trainer_candidates = [
    Path(PROJECT_DIR) / "notebooks" / "trainer.py",
    Path(PROJECT_DIR) / "trainer.py",
]
custom_trainer = next((p for p in trainer_candidates if p.exists()), None)
if custom_trainer is None:
    from google.colab import files
    print("Không tìm thấy trainer.py trong repo project. Upload trainer.py custom KD.")
    uploaded = files.upload()
    if "trainer.py" not in uploaded:
        raise FileNotFoundError("Cần upload trainer.py custom KD")
    custom_trainer = Path(PROJECT_DIR) / "notebooks" / "trainer.py"
    custom_trainer.parent.mkdir(parents=True, exist_ok=True)
    shutil.move("trainer.py", custom_trainer)

target_trainer = Path(ULTRALYTICS_KD_DIR) / "ultralytics" / "engine" / "trainer.py"
assert custom_trainer.exists(), f"Không tìm thấy custom trainer: {custom_trainer}"
assert target_trainer.parent.exists(), f"Không tìm thấy thư mục target: {target_trainer.parent}"
shutil.copy2(custom_trainer, target_trainer)
print("Copied custom KD trainer to:", target_trainer)

# Cài editable sau cùng để training_pipeline/src/train.py import đúng bản KD,
# kể cả khi requirements trước đó đã kéo Ultralytics gốc.
run(f"pip install -e {ULTRALYTICS_KD_DIR}")

run("python -c 'import ultralytics, inspect; from ultralytics import YOLO; import ultralytics.engine.trainer as t; print(\"Ultralytics path:\", ultralytics.__file__); print(\"Trainer path:\", t.__file__); print(\"Ultralytics KD OK\")'")

## 5. Cấu hình Kaggle

Cách khuyến nghị: vào Kaggle > Account > Create New API Token, tải `kaggle.json`, rồi upload ở cell dưới.

In [ ]:
from pathlib import Path

kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
if not kaggle_json.exists():
    from google.colab import files
    print("Upload kaggle.json")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Bạn cần upload file kaggle.json")
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    shutil.move("kaggle.json", kaggle_json)
    kaggle_json.chmod(0o600)

print("Kaggle config:", kaggle_json)
run("kaggle --version")

## 6. Tải dataset từ Kaggle

In [ ]:
Path(RAW_DATA_DIR).mkdir(parents=True, exist_ok=True)

# Ưu tiên dùng CLI của repo để organize dataset. Nếu lỗi, fallback sang kaggle CLI.
cmd_repo = (
    f"python -m data_pipeline kaggle download "
    f"--dataset {KAGGLE_DATASET} "
    f"--output {RAW_DATA_DIR} "
    f"--organize"
)
result = subprocess.run(cmd_repo, shell=True, cwd=PROJECT_DIR)

if result.returncode != 0:
    print("Repo data pipeline failed. Fallback to Kaggle CLI.")
    run(f"kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_DATA_DIR} --unzip")

run(f"find {RAW_DATA_DIR} -maxdepth 4 -type f | head -60", check=False)

## 7. Tạo subset và xác định `data.yaml`

In [ ]:
import yaml

def find_data_yaml(root):
    root = Path(root)
    candidates = list(root.rglob("data.yaml")) + list(root.rglob("dataset.yaml"))
    if not candidates:
        return None
    # Ưu tiên file gần root nhất.
    return sorted(candidates, key=lambda p: len(p.parts))[0]

DATA_ROOT = RAW_DATA_DIR

if USE_SUBSET:
    Path(SUBSET_DATA_DIR).parent.mkdir(parents=True, exist_ok=True)
    cmd_subset = (
        f"python -m data_pipeline dataset subset "
        f"--input {RAW_DATA_DIR} "
        f"--output {SUBSET_DATA_DIR} "
        f"--size {SUBSET_SIZE}"
    )
    subset_result = subprocess.run(cmd_subset, shell=True, cwd=PROJECT_DIR)
    if subset_result.returncode == 0:
        DATA_ROOT = SUBSET_DATA_DIR
    else:
        print("Subset creator failed. Continue with raw dataset.")

data_yaml = find_data_yaml(DATA_ROOT) or find_data_yaml(RAW_DATA_DIR)

if data_yaml is None:
    # Fallback khi dataset đã có cấu trúc train/valid/test nhưng thiếu YAML.
    default_names = ["bicycle", "bus", "car", "motorbike", "person"]
    data_yaml = Path(DATA_ROOT) / "data.yaml"
    config = {
        "path": str(Path(DATA_ROOT).resolve()),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "names": default_names,
        "nc": len(default_names),
    }
    data_yaml.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

print("DATA_ROOT:", DATA_ROOT)
print("DATA_YAML:", data_yaml)
print(Path(data_yaml).read_text()[:2000])

## 8. Train teacher model và xuất `teacher_best.pt`

In [ ]:
teacher_cmd = (
    f"python scripts/train_teacher_model.py "
    f"--data {data_yaml} "
    f"--model {TEACHER_MODEL} "
    f"--epochs {TEACHER_EPOCHS} "
    f"--batch {TEACHER_BATCH_SIZE} "
    f"--imgsz {IMG_SIZE} "
    f"--device {DEVICE} "
    f"--project runs/teacher "
    f"--name teacher_yolo "
    f"--no-mlflow"
)
run(teacher_cmd, cwd=PROJECT_DIR)

teacher_candidates = []
for root in [Path(PROJECT_DIR), Path(ULTRALYTICS_KD_DIR)]:
    teacher_candidates.extend(root.glob("runs/**/teacher_yolo*/weights/best.pt"))
    teacher_candidates.extend(root.glob("runs/teacher/**/weights/best.pt"))

teacher_candidates = sorted(
    {p.resolve() for p in teacher_candidates if p.exists()},
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
teacher_best = teacher_candidates[0] if teacher_candidates else None

print("teacher_best:", teacher_best)
assert teacher_best and teacher_best.exists(), "Không tìm thấy teacher best.pt"

## 9. Train student baseline và xuất `student_best.pt`

In [ ]:
from ultralytics import YOLO

student = YOLO(STUDENT_MODEL)
student_results = student.train(
    data=str(data_yaml),
    epochs=STUDENT_EPOCHS,
    batch=STUDENT_BATCH_SIZE,
    imgsz=IMG_SIZE,
    device=DEVICE,
    project="runs/student_baseline",
    name="student_yolo",
    save=True,
    plots=True,
)

student_best = Path(student_results.save_dir) / "weights" / "best.pt"
print("student_best:", student_best)
assert student_best.exists(), "Không tìm thấy student best.pt"

## 10. Train student bằng Knowledge Distillation

Cell này chạy KD mặc định. Nếu lỗi `unexpected keyword argument 'teacher'` hoặc `distillation_config_loss`, nghĩa là môi trường Ultralytics chưa có KD trainer tương thích và cần bổ sung phần trainer trước khi tiếp tục.

In [ ]:
kd_best = None

assert RUN_KD_OPTIONAL is True, "Notebook đang được cấu hình để chạy KD luôn"

# Tạo config KD riêng cho Colab để epochs/batch/device luôn khớp với cell CONFIG.
base_kd_config = Path(PROJECT_DIR) / "training_pipeline/src/config/train.yaml"
colab_kd_config = Path(PROJECT_DIR) / "training_pipeline/src/config/train_colab.yaml"
with base_kd_config.open("r", encoding="utf-8") as f:
    kd_cfg = yaml.safe_load(f)
kd_cfg.setdefault("training", {})
kd_cfg["training"]["epochs"] = STUDENT_EPOCHS
kd_cfg["training"]["batch"] = STUDENT_BATCH_SIZE
kd_cfg["training"]["imgsz"] = IMG_SIZE
kd_cfg["training"]["device"] = DEVICE
kd_cfg.setdefault("logging", {})
kd_cfg["logging"]["project"] = "runs/distillation"
kd_cfg["logging"]["name"] = "student_kd"
colab_kd_config.write_text(yaml.safe_dump(kd_cfg, sort_keys=False), encoding="utf-8")
print("KD config:", colab_kd_config)

kd_cmd = (
    f"python training_pipeline/src/train.py "
    f"{colab_kd_config} "
    f"--teacher-weights {teacher_best} "
    f"--student-weights {STUDENT_MODEL} "
    f"--data {data_yaml} "
    f"--mlflow-tracking-uri file:/content/mlruns "
    f"--mlflow-experiment traffic-distillation "
    f"--mlflow-run-name student_kd"
)
run(kd_cmd, cwd=PROJECT_DIR)

kd_candidates = []
for root in [Path(PROJECT_DIR), Path(ULTRALYTICS_KD_DIR)]:
    kd_candidates.extend(root.glob("runs/**/student_kd*/weights/best.pt"))
    kd_candidates.extend(root.glob("runs/distillation/**/weights/best.pt"))
kd_candidates = sorted(
    {p.resolve() for p in kd_candidates if p.exists()},
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
kd_best = kd_candidates[0] if kd_candidates else None
print("kd_best:", kd_best)
assert kd_best and Path(kd_best).exists(), "KD chạy xong nhưng không tìm thấy student KD best.pt"

## 11. Gom artifacts `.pt`

In [ ]:
artifact_dir = Path(ARTIFACT_DIR)
artifact_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(teacher_best, artifact_dir / "teacher_best.pt")
shutil.copy2(student_best, artifact_dir / "student_best.pt")

# Model dùng cho serving: ưu tiên KD nếu có, nếu không dùng student baseline.
serving_source = kd_best if kd_best and Path(kd_best).exists() else student_best
shutil.copy2(serving_source, artifact_dir / "serving_model.pt")

if kd_best and Path(kd_best).exists():
    shutil.copy2(kd_best, artifact_dir / "student_kd_best.pt")

run(f"ls -lh {ARTIFACT_DIR}", check=False)

## 12. Validate nhanh và export ONNX optional

In [ ]:
# Validate nhanh model dùng cho serving. Ghi lại mAP/FPS/latency từ output cell này để đưa vào báo cáo.
run(f"yolo detect val model={artifact_dir / 'serving_model.pt'} data={data_yaml} imgsz={IMG_SIZE} device={DEVICE}", check=False)

# Export ONNX nếu cần. File .onnx cũng sẽ được copy vào artifacts nếu export thành công.
export_result = subprocess.run(
    f"yolo export model={artifact_dir / 'serving_model.pt'} format=onnx imgsz={IMG_SIZE}",
    shell=True,
    cwd=PROJECT_DIR,
)
if export_result.returncode == 0:
    onnx_candidates = sorted(artifact_dir.glob("*.onnx")) + sorted(Path(PROJECT_DIR).glob("*.onnx"))
    for onnx_file in onnx_candidates:
        if onnx_file.exists():
            shutil.copy2(onnx_file, artifact_dir / onnx_file.name)

run(f"ls -lh {ARTIFACT_DIR}", check=False)

## 13. Nén artifacts và tải xuống

In [ ]:
zip_path = "/content/model_artifacts.zip"
if Path(zip_path).exists():
    Path(zip_path).unlink()

run(f"cd /content && zip -r {zip_path} model_artifacts")

from google.colab import files
files.download(zip_path)

## Files cần dùng sau khi train

Sau khi chạy xong, thư mục `/content/model_artifacts` sẽ có tối thiểu:

- `teacher_best.pt`: model teacher.
- `student_best.pt`: model student baseline.
- `serving_model.pt`: model dùng cho FastAPI/Gradio.

Nếu chạy KD thành công sẽ có thêm:

- `student_kd_best.pt`.

Nếu export ONNX thành công sẽ có thêm file `.onnx`.